# Cold Mail Generator

Step-by-step notebook for the general cold-email pipeline (Groq + LangChain + ChromaDB). For the interactive app, run `streamlit run main.py`.

In [ ]:
import os

from dotenv import load_dotenv
from langchain_groq import ChatGroq

load_dotenv()
GROQ_API_KEY = os.getenv("GROQ_API_KEY")
if not GROQ_API_KEY:
    raise ValueError("Set GROQ_API_KEY in a .env file (never commit API keys).")

In [ ]:
llm = ChatGroq(
    temperature=0,
    groq_api_key=GROQ_API_KEY,
    model_name="llama-3.1-70b-versatile",
)

In [ ]:
from langchain_community.document_loaders import WebBaseLoader

loader = WebBaseLoader("https://jobs.nike.com/job/R-33460")
page_data = loader.load().pop().page_content
print(page_data)

In [ ]:
from langchain_core.prompts import PromptTemplate

prompt_extract = PromptTemplate.from_template(
        """
        ### SCRAPED TEXT FROM WEBSITE:
        {page_data}
        ### INSTRUCTION:
        The scraped text is from a careers or job posting page.
        Extract the job posting and return JSON with keys: `role`, `experience`, `skills`, `description`.
        Only return the valid JSON.
        ### VALID JSON (NO PREAMBLE):    
        """
)

chain_extract = prompt_extract | llm 
res = chain_extract.invoke(input={'page_data':page_data})
type(res.content)

In [ ]:
from langchain_core.output_parsers import JsonOutputParser

json_parser = JsonOutputParser()
json_res = json_parser.parse(res.content)
json_res

In [ ]:
type(json_res)

In [ ]:
import pandas as pd

df = pd.read_csv("my_portfolio.csv")
df

In [ ]:
import uuid
import chromadb

client = chromadb.PersistentClient('vectorstore')
collection = client.get_or_create_collection(name="portfolio")

if not collection.count():
    for _, row in df.iterrows():
        collection.add(documents=row["Techstack"],
                       metadatas={"links": row["Links"]},
                       ids=[str(uuid.uuid4())])

In [ ]:
links = collection.query(query_texts=job['skills'], n_results=2).get('metadatas', [])
links

In [ ]:
job

In [ ]:
job = json_res
job['skills']

In [ ]:
SENDER_NAME = "Alex"
COMPANY_NAME = "Your Company"
COMPANY_DESCRIPTION = (
    "An AI and software consulting company that helps businesses integrate automated tools, "
    "scale operations, optimize processes, and improve efficiency through tailored solutions."
)

prompt_email = PromptTemplate.from_template(
    """
    ### JOB DESCRIPTION:
    {job_description}

    ### INSTRUCTION:
    You are {sender_name}, a business development professional at {company_name}.
    {company_name} is described as: {company_description}

    Write a professional, personalized cold email to the hiring team about the role above.
    Explain how {company_name} can help meet their needs.
    Include the most relevant portfolio links from this list: {link_list}

    Write as {sender_name} from {company_name}.
    Do not include a preamble—output only the email (subject line optional).
    ### EMAIL (NO PREAMBLE):
    """
)

chain_email = prompt_email | llm
res = chain_email.invoke(
    {
        "job_description": str(job),
        "link_list": links,
        "sender_name": SENDER_NAME,
        "company_name": COMPANY_NAME,
        "company_description": COMPANY_DESCRIPTION,
    }
)
print(res.content)